# 12주차 · 발사서비스 시장과 경제학
### 수요는 누가 만들고, 가격은 무엇이 결정하는가 — TRANSCOST 비용모형과 시장구조 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabraxas/LVs-and-Policy/blob/main/lecture/week12/week12.ipynb)

> **우주수송정책과 발사체 기술** — Week 12
> 사업기획보고서 **「③ 시장분석」** 파트의 정량 도구 · **11주차** BMC '비용구조·수익원' 블록의 정량화

> 가격·학습곡선 소스: D.E. Koelle, *Handbook of Cost Engineering for Space Transportation Systems* — TRANSCOST 8.2 (TCS, 2013); Drenthe et al. (2017) EUCASS; Stappert et al. (2022) DLR

> 이 노트북은 GitHub에 저장되고 Colab에서 실행됩니다. 위 배지를 눌러 Colab에서 열거나, 아래 첫 코드 셀부터 순서대로 실행하세요.

In [ ]:
# ▶ 실행 전 준비 — 이 셀을 먼저 실행하세요 (약 10~20초 소요)
!pip install -q ipywidgets
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/_shared/course_interactive.py",
    "course_interactive.py")

from course_interactive import *
setup_korean_font()
print("준비 완료 — 아래 셀들을 순서대로 실행하며 강의를 진행하세요.")


## 학습 목표

- [ ] **수요 구조를 파악함** — 발사서비스 수요의 3원천(정부·상업·군집)을 구분하고, 각 세그먼트의 구매 기준과 규모를 설명한다
- [ ] **TRANSCOST 비용모형을 습득함** — CER·학습곡선·발사빈도(LpA)로 발사원가(CpF)와 가격(PpF)을 추정하는 절차를 익힌다
- [ ] **시장구조 변동을 분석함** — 라이드셰어의 구조적 충격과 소형발사체 생태계의 부침(2018→2025)을 경제 논리로 해석한다
- [ ] **시장분석 파트를 작성함** — 사업기획보고서 '③ 시장분석'을 TAM-SAM-SOM 체계와 가격 벤치마크로 작성할 준비를 마친다

## 오늘 강의의 3부 구성: 수요 → 비용·가격 → 시장구조

지난주 BMC에서 '수익원'과 '비용구조'는 질문으로만 남겨두었다. 오늘은 그 두 블록을 숫자로 채운다 — 누가 사는지(수요), 얼마에 만들 수 있는지(TRANSCOST), 그리고 시장이 그 가격을 어떻게 재편해 왔는지(구조).

| 부 | 주제 |
|---|---|
| 1부 | 수요: 누가 사는가 — 정부·상업·군집 3대 수요원의 구매 기준과 시장 규모 |
| 2부 | 비용·가격: TRANSCOST — CER·학습곡선·발사빈도(LpA) → 발사원가(CpF)와 가격(PpF)의 결정 구조 |
| 3부 | 시장구조: 재편의 동학 — 라이드셰어 충격과 소형발사체 생태계 부침 + 시장분석 파트 작성 실습 |

## PART 1 · 발사서비스 수요의 3원천 — 구매 기준이 서로 다르다

시장분석의 첫 단추는 '발사 수요'를 하나의 시장으로 뭉뚱그리지 않는 것이다. 세 수요원은 규모·가격민감도·구매기준이 근본적으로 다르며, 어떤 세그먼트를 겨냥하는지에 따라 사업모델 전체가 달라진다.

| 구분 | 정부·안보 수요 | 상업 위성사업자 | 군집(메가컨스텔레이션) |
|---|---|---|---|
| 대표 고객 | 군·정보기관, 우주청, 과학임무 | GEO 통신·방송, 지구관측 기업 | Starlink, Kuiper, OneWeb 등 |
| 구매 기준 | 신뢰성·보안·자국발사 요건이 가격보다 우선 | 가격+일정+궤도 정합성의 균형 | 압도적 물량 — $/kg 최우선, 자체발사 성향 |
| 가격 민감도 | 낮음 (프리미엄 지불 용의) | 중간 | 매우 높음 (수직계열화 유인) |
| 시장 특성 | 안정적·장기계약, 진입장벽=인증·신뢰 | 경기·위성 수명주기에 연동 | 소수 사업자가 물량 대부분을 좌우 |

> **보고서 적용**: '③ 시장분석'은 반드시 목표 세그먼트를 하나 선언하고 시작할 것 — **11주차** BMC의 '고객 세그먼트' 블록과 동일한 답이어야 한다.

## PART 2 · TRANSCOST — 발사 비용을 추정하는 표준 공개 모델

독일 D.E. Koelle가 정립한 TRANSCOST는 발사체 비용추정 분야에서 가장 널리 쓰이는 공개 파라메트릭 모델이다. 질량 등 소수의 설계변수로부터 하향식(top-down)으로 비용을 추정한다.

| 특징 | 내용 |
|---|---|
| Work-Year(WYr) 단위 | 비용을 인플레이션·환율에 무관한 '연간 인건비' 단위로 표현 — 수십 년에 걸친 기체 간 비교 가능 |
| 3단계 비용 구조 | 개발(DEV) · 생산(MAN) · 운영(OPS)을 각각 별도의 추정식으로 계산 후 합산 |
| 질량 기반 CER | Cost = a·M^x 형태의 지수회귀식(비용추정관계식)을 액체단·고체단·엔진 등 범주별로 제공 |
| 보정계수 체계 | 기술성숙도(f1)·기술계수(f2)·팀 경험(f3)·국가생산성(f8)·상업화(f11) 등으로 현실 보정 |

> **BMC 연결**: TRANSCOST의 DEV·MAN·OPS가 곧 캔버스 '비용구조' 블록의 정량 버전이다. 지난주 질문으로 남긴 블록을 오늘 숫자로 채운다.

## PART 2 · 비용추정관계식(CER) — 질량에서 개발비로

$$C_{Dev} = f_1 \cdot f_2 \cdot f_3 \cdot a \cdot M^x \qquad (M = \text{시스템 건조질량})$$

f1 = 기술성숙도/복잡도, f2 = 구성요소 기술계수, f3 = 팀 경험 — Koelle가 제시한 권장범위 내에서 선정

| 구성요소 범주 | 질량 변수 M | 예시 계수(a, x)* |
|---|---|---|
| 액체 추진 소모성 1단 | 무추진제 건조질량 | a ≈ 100, x ≈ 0.555 (TRANSCOST 원본) |
| 날개형(유익) 재사용 1단 | 엔진 제외 건조질량 | a ≈ 1442, x ≈ 0.326 (FESTIP 기반) |
| 엔진(터보펌프식) | 엔진 건조질량 | 범주별 별도 CER (압력식/터보식 구분) |

*계수는 컴포넌트 범주·데이터셋에 따라 다름(Stappert et al. 2022 재현치). 실무 적용 시 반드시 최신 TRANSCOST 매뉴얼 표를 참조 — 수업에서는 **'지수 x < 1'**, 즉 규모의 경제가 질량에도 작용한다는 직관이 핵심.

**[인터랙티브] CER 개발비 계산기** — 구성요소·질량·보정계수를 조작해 개발비가 어떻게 산정되는지 확인

In [ ]:
cer_development_cost_calculator()

## PART 2 · 학습곡선 — 반복 생산이 단위비용을 낮춘다

**Crawford 단위 학습곡선** (TRANSCOST 계열 연구의 표준)

$$U_n = T_1 \cdot n^b, \qquad b = \ln(p)/\ln(2)$$

T1 = 이론적 1호기 비용, n = 누적 생산 순번, p = 학습률(발사체 관행: 90%) → 누적생산 2배마다 단위비용이 p배로

예시(p=90%): 2호기 = T1의 90% → 4호기 81% → 8호기 약 73% → 64호기 약 53%

> **민감도 경고**: 학습률을 82~96% 범위에서 잘못 가정하면 생산비 추정이 +39%~−54%까지 왜곡될 수 있음(RAND) — 보고서에서 학습률 가정을 반드시 명시할 것

> **전략적 함의**: '많이 만드는 자가 싸게 만든다' — 대량생산·고빈도 사업자(SpaceX)의 원가 우위는 기술이 아니라 이 산수에서 나온다

**[인터랙티브] Crawford 학습곡선 탐색기** — T1·학습률·순번을 조작해 단위비용 하락을 확인

In [ ]:
learning_curve_explorer()

64호기 기준: 95%면 T1의 약 74%, 90%면 약 53%, 85%면 약 37%

- 재사용은 이 곡선을 '생산'에서 '운용'으로 옮긴 것 — 같은 기체의 재비행 정비비도 학습곡선을 탄다 (**5주차** 연계)
- 누적 수십 기를 전제할 수 없는 신생기업은 T1 근처의 높은 원가에서 출발 — 가격이 아닌 가치제안 차별화가 필요한 이유 (**11주차** 연계)

## PART 2 · 운영비용 — 직접(DOC)과 간접(IOC), 그리고 발사빈도

TRANSCOST는 운영비를 발사 1회에 직결되는 직접운영비(DOC)와, 발사 유무와 무관하게 조직 유지에 드는 간접운영비(IOC)로 나눈다. Koelle는 연간발사횟수(LpA)를 운영비의 가장 강력한 결정변수로 지목했다.

| 구분 | 구성 요소 | 핵심 관계식(개념) |
|---|---|---|
| 직접운영비(DOC) | 지상운영·추진제·비행운영·수송회수·수수료·보험 | 지상운영 ∝ W·M₀^0.67·L^-0.9·N^0.7·(보정계수들) |
| 간접운영비(IOC) | 행정·마케팅·기술지원 등 조직 유지비 | IOC = 40·S + 22.5·LpA^-0.379·W |

변수(요약): W=Work-Year 단가, M₀=이륙질량, L(LpA)=연간발사횟수, N=단수, S=외주비중. 관계식의 정확한 형태·계수는 TRANSCOST 매뉴얼 참조 — 핵심은 **IOC가 LpA의 음의 거듭제곱으로 떨어진다**는 구조다. 발사가 뜸한 조직일수록 발사 1회가 짊어지는 간접비가 커진다.

## PART 2 · 발사빈도(LpA)의 위력 — 간접비의 급락

- LpA 1→50이면 회당 간접비 지수가 100→약 23으로 급락
- Drenthe et al.(2017): 연 4회 이상부터 운영비가 반복비용 대비 안정화되는 경향
- 학습곡선(생산)과 LpA(운영)는 곱으로 작용 — '많이 만들고 자주 쏘는' 사업자만 이중의 원가 우위를 누린다

> **보고서 적용**: 목표 LpA와 그 근거(수주 계획)를 명시하지 않은 원가 추정은 신뢰받지 못한다

**[인터랙티브] LpA·간접운영비 탐색기**

In [ ]:
lpa_indirect_cost_explorer()

## PART 2 · 발사원가(CpF)에서 가격(PpF)으로 — 수익원 블록의 정량화

| 요소 | 내용 |
|---|---|
| 개발비 상각 | DEV / Na — 총 개발비를 계획 발사횟수 Na로 회당 배분 |
| 생산비 | MANₙ (학습곡선) — n호기 생산비, 누적생산에 따라 하락 |
| 운영비 | OPSₙ = DOC + IOC — 발사빈도(LpA)에 민감 |

$$CpF_n = \frac{DEV}{N_a} + MAN_n + OPS_n \qquad\longrightarrow\qquad PpF_n = CpF_n \times (1 + \text{이윤율, 예: 8\%})$$

**검증 사례 (Drenthe et al. 2017)**

| 항목 | 추정치 | 실측치 | 오차 |
|---|---|---|---|
| Falcon 1 개발비(2010$) | $96M | $90M | +7% |
| Falcon 9 개발비(2010$) | $419M | $372M | +13% |
| Falcon 1 가격, LpA=10(2008$) | $8.7M | $7.9M | +10% |
| Pegasus XL 가격, LpA=12(2015€) | 20.0M€ | 20.3M€ | +1.5% |

> 단, 수직계열화 상업기업엔 표준 CER이 과대추정 경향(Falcon 9 재사용 개발비를 10배 이상 과대평가한 사례, Stappert 2022) — 상업 보정계수(f11) 적용과 함께 근본적 한계를 인지할 것.

**[인터랙티브] CpF → PpF 종합 계산기**

In [ ]:
cpf_to_ppf_calculator()

## PART 3 · 라이드셰어 충격 — 규모의 경제를 '나눠 파는' 사업

2019년 시작된 SpaceX Transporter는 대량생산 Falcon 9의 학습곡선·LpA 효과를 소형위성 고객에게 kg 단위로 분배했다. 소형위성 발사의 실효가격이 한 자릿수 천 달러/kg대로 내려앉으며 시장 구조가 재편됐다.

- 가격 5배 하락의 원천은 신기술이 아니라 **'남의 학습곡선에 합승'**하는 구조
- 2026년 초 기준 약 $7,000/kg 수준으로 재상승(수요 증가·인플레이션 반영), 그래도 전용발사 대비 수 분의 1
- 전용 소형발사(Electron 등)는 2~4만$/kg 프리미엄으로 '내 궤도·내 일정' 가치에 집중
- → 시장은 **'저가 합승'과 '고가 전용'**으로 양분 — 중간지대가 소멸

**[그림] 라이드셰어 vs 전용발사 가격 비교** — 아래 코드 셀을 실행해 확인

In [ ]:
rideshare_vs_dedicated_chart()

## PART 3 · 소형발사체 생태계의 부침 — 과잉투자 → 조정 → 재편

| 연도 | 국면 | 내용 |
|---|---|---|
| 2018 | 과잉 투자기 | 100여 개 스타트업 난립. '소형위성엔 전용발사'라는 서사에 VC 자금 쇄도 — 수요 추정은 낙관 일색 |
| 2022 | 조정·붕괴기 | SPAC 상장사 다수 몰락, Virgin Orbit 파산(2023). 라이드셰어 실효가격에 사업계획의 전제가 무너짐 |
| 2025~26 | 재편·생존기 | Rocket Lab 등 소수만 흑자 — 전용발사 프리미엄 + 우주시스템 확장으로 생존. 나머지는 통폐합·철수 |

**경제학적 해석 — 오늘 배운 도구로 다시 읽기**

실패한 사업계획들의 공통점: ① 학습곡선의 T1 근처(고원가)에서 출발하면서도, ② 저빈도(LpA 소수)라 간접비 부담이 극대화되고, ③ 가격은 라이드셰어와 비교당하는 구조. 세 변수(T1·LpA·가격 벤치마크)를 정직하게 놓고 보면 2022년의 붕괴는 예견 가능했다 — 여러분의 시장분석 파트가 해야 할 일이 정확히 이것이다.

## 실습 · 사업기획보고서 「③ 시장분석」 작성 4단계

지난주 BMC 초안에서 정한 고객 세그먼트를 이어받아, 오늘 배운 도구로 시장분석 파트의 뼈대를 완성한다 (팀별 60분).

**① 수요 정량화: TAM → SAM → SOM (15분)** — 전체 발사수요(TAM)에서 목표 세그먼트·궤도로 좁힌 유효시장(SAM), 현실적 수주 가능분(SOM)을 연간 발사횟수·질량으로 추정. 근거 출처(위성 발사계획, 정부 중기계획)를 명시할 것.

**[인터랙티브] TAM-SAM-SOM 퍼널 계산기**

In [ ]:
tam_sam_som_funnel()

**② 가격 벤치마크 설정 (15분)** — 경쟁 대안의 실효가격을 표로 정리: 라이드셰어 $/kg, 경쟁 전용발사 회당 가격, 정부 전용임무 낙찰가. 우리 가격(PpF)이 어느 축에서 경쟁하는지 선언.

**③ 원가 검증: CpF 어림 (15분)** — TRANSCOST 논리로 DEV/Na + MAN(학습률 명시) + OPS(목표 LpA 명시)를 어림하고, ②의 가격에서 역산한 허용원가와 비교 — 격차가 크면 사업모델을 수정. 위 CpF→PpF 계산기를 활용할 것.

**④ 구조 리스크 점검 (15분)** — 2018~22년 소형발사체 붕괴의 세 변수(T1 고원가·저LpA·라이드셰어 벤치마크)에 우리 계획을 대입 — 같은 함정에 빠지지 않는 논거를 리스크 파트(⑥)로 넘길 것.

## 정책적 시사점과 토론

> **핵심 시사점**: 발사 원가경쟁력은 보조금이 아니라 학습곡선과 발사빈도라는 '누적의 산수'에서 나온다. 따라서 발사체 기업 육성 정책의 요체는 일회성 개발비 지원이 아니라, 기업이 그 산수를 굴릴 수 있도록 연속 물량(정부 앵커수요)과 상업 수요 접근(발사허가·발사장·라이드셰어 역량)을 설계해 주는 것이다.

1. 한국의 연간 위성 발사수요만으로 국내 발사체 기업이 손익분기 LpA에 도달할 수 있는가? 부족분은 어디서 채워야 하는가?
2. 라이드셰어가 지배하는 소형위성 시장에서, 정부가 국내 기업에 물량을 몰아주는 정책은 언제 정당화되고 언제 왜곡이 되는가?
3. TRANSCOST가 수직계열화 기업의 비용을 과대추정한다면, 정부의 사업 타당성 심사(예타 등)는 어떤 비용모형을 써야 하는가?

## 참고문헌

**[가격·학습곡선 주교재]** Koelle, D.E., *Handbook of Cost Engineering for Space Transportation Systems with TRANSCOST 8.2*, TCS-TR-200, TransCostSystems, Ottobrunn, 2013.

**보조 참고자료**
- Drenthe, N.T., Zandbergen, B.T.C., van Pelt, M.O. (2017), "Cost Estimating of Commercial Smallsat Launch Vehicles," EUCASS2017-286
- Stappert, S. et al. (2022), "Evaluation of Parametric Cost Estimation in the Preliminary Design Phase of Reusable Launch Vehicles," 9th EUCASS, DLR
- Koelle, D.E. (1984), "The TRANSCOST-Model for Launch Vehicle Cost Estimation," *Acta Astronautica*, Vol. 11
- SpaceNews·New Space Economy 시장 분석 (2025~26) — 라이드셰어 가격, 소형발사체 재편 동향

> **다음 주(13주차) 예고**: 스타십과 게임체인저 기술 — 오늘의 학습곡선·LpA 논리를 극한까지 밀어붙인 초대형 완전재사용이 시장과 정책에 던지는 충격.